In [2]:
import numpy as np 
import h5py as h5 
import matplotlib.pyplot as plt

# plotting config
import matplotlib
matplotlib.rcParams.update({'text.usetex': True, 
                            'font.family': 'Computer Modern Roman'})

from scipy.interpolate import UnivariateSpline
from scipy.optimize import minimize

from compact.lpt import *
from compact.streaming import *
from compact.model import *
from compact.plot import *


from halotools.mock_observables import tpcf_multipole


from tqdm import tqdm
import emcee

from scipy.interpolate import UnivariateSpline

import warnings
warnings.filterwarnings('ignore')

# plotting config
import matplotlib
matplotlib.rcParams.update({'text.usetex': True, 
                            'font.family': 'Computer Modern Roman'})
plt.rcParams['axes.labelsize'] = 20


import colossus
import pandas as pd

# reload project when changes are made
%load_ext autoreload
%autoreload 2 

In [4]:
dex_width = 0.1  # bin width in log10(mass); narrower = 0.1, wider = 0.3
log_edges = np.array([[12, 12.01], [12.5, 12.6], [13, 13.1], [13.5, 13.6], [14.0, 14.2], [14.4, 14.7]])
massbins = 10**log_edges
print(np.log10(massbins))

[[12.   12.01]
 [12.5  12.6 ]
 [13.   13.1 ]
 [13.5  13.6 ]
 [14.   14.2 ]
 [14.4  14.7 ]]


In [6]:
massbins.shape

(6, 2)

In [7]:
# Check halo catalogs
halo_path = f'/spiff/hengweichang/AbacusSummit_base/snapshots/AbacusSummit_base_c000_ph000/halos/z0.200/halo_combined.h5'

box_length = 2000 #Mpc/h, AS box side length

m_p = 2e9 # h^-1 M_sun, particle mass

CHUNK_SIZE = 100_000
matching_rows = np.zeros(massbins.shape[0])

with h5.File(halo_path, 'r') as hcat:
    dset = hcat['halos']

    print(dset.dtype.names)

    total_rows = dset.shape[0] 

    for i in range(0, total_rows, CHUNK_SIZE):
        # Extract the specific column chunk
        chunk = dset[i : i + CHUNK_SIZE]['N_total'] * m_p
        
        # Apply your mask and sum the Trues
        # Example condition: column value is greater than 50
        for k in range(massbins.shape[0]):
            matching_rows[k] += np.sum((chunk >= massbins[k, 0])&(chunk <= massbins[k, 1]))   

print(matching_rows)     

    # hm_cut = hcat['Mvir'][()] >= 1e13

    # hpos = np.array([
    #     hcat['X'][()][hm_cut],
    #     hcat['Y'][()][hm_cut], 
    #     hcat['Z'][()][hm_cut],
    # ]).T 

    # hvel = np.array([
    #     hcat['VX'][()][hm_cut],
    #     hcat['VY'][()][hm_cut], 
    #     hcat['VZ'][()][hm_cut],
    # ]).T

    # hid = hcat['ID'][()][hm_cut]
    # upid = hcat['PID'][()][hm_cut]

('hid_cleaned', 'hid_uncleaned', 'N_total', 'N_merged', 'N_L1', 'x', 'y', 'z', 'vx', 'vy', 'vz', 'sigmav3d_L2com', 'r98_com_i16', 'r98_L2com_i16')
[ 698889. 2108413.  689460.  197828.   74035.   17111.]


In [16]:
halo_path_test = f'/spiffball/edgarmsc/simulations/Quijote/Halos/Rockstar/fiducial_HR/0/hlist_4.hdf5'

q_hnum = np.zeros(massbins.shape[0])
with h5.File(halo_path_test, 'r') as hcat:    
    
    for k in range(len(massbins)):
        mcut = (hcat['Mvir'][()] >= massbins[k][0]) & (hcat['Mvir'][()] <= massbins[k][1]) # h^-1 M_sun
        q_hnum[k] = np.sum(mcut)
print(q_hnum)

[     0.   8047. 104281.  33413.  15524.   4823.]


In [17]:
2108413/8047

262.0123027215111

In [9]:
17111/4823

3.5477918308106986

In [11]:
74035/15524

4.76906725070858

In [12]:
689460/104281

6.611559152674025

In [13]:
197828/33413

5.920689551970789

In [30]:
# check cosmo

test_dat = f'/spiffball/edgarmsc/simulations/Quijote/Snapshots/latin_hypercube_HR_cosmo/Cosmo_params_{sim_num}.dat'

df = pd.read_csv(test_dat, sep=r'\s+', header=None, names=['Om', 'Ob', 'h', 'ns', 's8'])

print(df.head())

params = {'flat': True, 'H0': 100 * df['h'][0], 'Om0': df['Om'][0], 'Ob0': df['Ob'][0], 'sigma8': df['s8'][0], 'ns': df['ns'][0]}

cosmo = cosmology.setCosmology('myCosmo', params)

       Om       Ob       h      ns      s8
0  0.3345  0.03171  0.7971  1.0785  0.6753


In [93]:
#massbins = np.array([[1e13, 2e13], [5e13, 6e13], [3e14, 1e15]])
sim_num = 1992

cosmo_nums = np.array([1992, 1994, 1927, 1986, 1852])

for sim_num in cosmo_nums:

    halo_path_test = f'/spiffball/edgarmsc/simulations/Quijote/Halos/Rockstar/latin_hypercube_HR/{sim_num}/hlist_4.hdf5'

    dex_width = 0.1  # bin width in log10(mass); narrower = 0.1, wider = 0.3
    log_edges = np.arange(np.log10(1e13), np.log10(1e15) + dex_width, dex_width)
    massbins = np.column_stack([10**log_edges[:-1], 10**log_edges[1:]])[::6] # select 4 bins from the full set of mass bins

    # print(massbins)

    with h5.File(halo_path_test, 'r') as hcat:    
        
        for k in range(len(massbins)):
            mcut = (hcat['Mvir'][()] >= massbins[k][0]) & (hcat['Mvir'][()] <= massbins[k][1]) # h^-1 M_sun
            print(np.sum(mcut))

26892
3621
216
0
71714
18159
3582
351
36117
6598
757
34
164538
63873
14044
1200
134521
40009
8976
998


In [79]:
#4/64 [03:41<54:11, 54.19s/it]
#64/64 [08:58<00:00,  7.01s/it]
# For 5 cosmologies and 3 mass bins, implies around 15 hrs (or ~ 2.5 hours with parallelization) to compute all the pairwise velocity selections for the Quijote simulations.
# let's select the cosmologies we'll use. 

available_sim_nums = np.array([                                                    # only this subset of the 2000 sims are available on spiffball
    1402, 1655, 1740, 1791, 1807, 1820, 1839, 1861, 1889, 1902, 1911, 1915, 1923,
    1930, 1936, 1945, 1952, 1962, 1966, 1974, 1979, 1984, 1988, 1992, 1996,
    1625, 1656, 1757, 1793, 1810, 1824, 1852, 1871, 1894, 1906, 1912, 1916, 1927,
    1933, 1937, 1948, 1955, 1963, 1967, 1975, 1980, 1985, 1989, 1993, 1997,
    1630, 1659, 1764, 1795, 1818, 1826, 1857, 1877, 1897, 1907, 1913, 1919, 1928,
    1934, 1943, 1949, 1956, 1964, 1970, 1976, 1982, 1986, 1990, 1994, 1998,
    1640, 1704, 1769, 1797, 1819, 1835, 1859, 1887, 1898, 1909, 1914, 1921, 1929,
    1935, 1944, 1950, 1960, 1965, 1972, 1977, 1983, 1987, 1991, 1995, 590,
])

sim_nums = [i for i in range(2_000)]

q_cosmos = np.zeros((len(sim_nums), 5))

for num in sim_nums:
    dat_path = f'/spiffball/edgarmsc/simulations/Quijote/Snapshots/latin_hypercube_HR_cosmo/Cosmo_params_{num}.dat'

    df = pd.read_csv(dat_path, sep=r'\s+', header=None, names=['Om', 'Ob', 'h', 'ns', 's8'])

    q_cosmos[num] = df.values[0]

q_cosmos_available = q_cosmos[available_sim_nums]


def select_lh_subset(params, k=5, rng=None, max_attempts=1000):
    """
    params : (N, d) array of cosmological parameters (e.g. Om, Ob, h, ns, s8)
    k      : number of sims to select == number of quantile bins per parameter
    Returns indices of k rows such that, for every parameter column, the k
    selected rows occupy k distinct quantile bins (a Latin-hypercube-like subset).
    """
    if rng is None:
        rng = np.random.default_rng()

    N, d = params.shape

    # assign each sim a quantile bin (0..k-1) per parameter
    bins = np.zeros((N, d), dtype=int)
    for j in range(d):
        edges = np.quantile(params[:, j], np.linspace(0, 1, k + 1))
        bins[:, j] = np.clip(np.searchsorted(edges, params[:, j], side='right') - 1, 0, k - 1)

    for _ in range(max_attempts):
        order = rng.permutation(N)
        used_bins = [set() for _ in range(d)]
        chosen = []

        for idx in order:
            row_bins = bins[idx]
            if all(row_bins[j] not in used_bins[j] for j in range(d)):
                chosen.append(idx)
                for j in range(d):
                    used_bins[j].add(row_bins[j])
                if len(chosen) == k:
                    return np.array(chosen), bins[chosen]

    raise RuntimeError(f"Couldn't find a valid Latin-hypercube-like subset in {max_attempts} attempts")

# params = np.loadtxt('latin_hypercube_params.txt')  # columns: Om, Ob, h, ns, s8
# selected_idx, selected_bins = select_lh_subset(params, k=5)
# print(selected_bins)  # each column should be a permutation of 0..4

selected_idx, selected_bins = select_lh_subset(q_cosmos_available, k=5)